# Tracking Political Change with Embeddings of Parliamentary Speeches
***
# Embedding Baseline Model (Jina)
## 1. Setup
### 1.1 Environment Setup

In [1]:
%pip install --index-url https://download.pytorch.org/whl/cu124 --extra-index-url https://pypi.org/simple \
    torch==2.6.0 torchvision torchaudio transformers==5.9.0 accelerate peft scikit-learn

Looking in indexes: https://download.pytorch.org/whl/cu124, https://pypi.org/simple
  Using cached https://download-r2.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp311-cp311-linux_x86_64.whl.metadata (28 kB)
  Using cached torchvision-0.28.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached torchaudio-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (6.9 kB)
  Using cached transformers-5.9.0-py3-none-any.whl.metadata (33 kB)
  Using cached accelerate-1.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.20.0-py3-none-any.whl.metadata (14 kB)
  Using cached filelock-3.32.3-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
  Using cached huggingface_hub-1.27.0-py3-none-any.whl.metadata (16 kB)
  Using cached regex-2026.7.19-cp311-cp311-manyli

### 1.2 Imports and Seeds

In [2]:
# Imports 
import pandas as pd
import numpy as np
import re
from transformers import AutoModel, AutoTokenizer
import torch
import torch.nn.functional as F
import time
import random
from embedding_utils import embed_speeches

In [3]:
SEED = 24
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


### 1.3 Model Loading

In [4]:
# Load the off-the-shelf Jina v3 model from Hugging Face
tokenizer = AutoTokenizer.from_pretrained("jinaai/jina-embeddings-v3-hf")
model = AutoModel.from_pretrained("jinaai/jina-embeddings-v3-hf").to(device)
model.eval()

Loading weights:   0%|          | 0/294 [00:00<?, ?it/s]

JinaEmbeddingsV3Model(
  (embeddings): JinaEmbeddingsV3Embeddings(
    (word_embeddings): Embedding(250002, 1024, padding_idx=1)
    (token_type_embeddings): Embedding(1, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (pooler): JinaEmbeddingsV3Pooler(
    (dense): Linear(in_features=1024, out_features=1024, bias=True)
    (activation): Tanh()
  )
  (rotary_emb): JinaEmbeddingsV3RotaryEmbedding()
  (layers): ModuleList(
    (0-23): 24 x JinaEmbeddingsV3Layer(
      (post_attention_layernorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (post_attention_dropout): Dropout(p=0.1, inplace=False)
      (post_mlp_dropout): Dropout(p=0.1, inplace=False)
      (mlp): JinaEmbeddingsV3MLP(
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
      )
      (self_attn): JinaE

## 2. Embedding
### 2.1 Load Corpus

In [5]:
path = "embedding_corpus.csv"
df = pd.read_csv(path)
print(f"Loaded {len(df)} speeches for baseline embedding.")

Loaded 82447 speeches for baseline embedding.


### 2.2 Generate Embeddings

In [6]:
texts = df["speechContent"].tolist()
print(f"Embedding {len(texts)} speeches with the baseline off-the-shelf encoder...")

t0 = time.time()

embeddings = embed_speeches(texts, model, tokenizer, device, batch_size=32,
                             checkpoint_every=200, checkpoint_path="baseline_checkpoint.npy")

elapsed = time.time() - t0

rate = len(texts) / elapsed
print(f"Shape: {embeddings.shape}")
print(f"Time: {elapsed/60:.1f} min | Rate: {rate:.2f} speeches/sec")

Embedding 82447 speeches with the baseline off-the-shelf encoder...
Embedded 32/82447
Embedded 64/82447
Embedded 96/82447
Embedded 128/82447
Embedded 160/82447
Embedded 192/82447
Embedded 224/82447
Embedded 256/82447
Embedded 288/82447
Embedded 320/82447
Embedded 352/82447
Embedded 384/82447
Embedded 416/82447
Embedded 448/82447
Embedded 480/82447
Embedded 512/82447
Embedded 544/82447
Embedded 576/82447
Embedded 608/82447
Embedded 640/82447
Embedded 672/82447
Embedded 704/82447
Embedded 736/82447
Embedded 768/82447
Embedded 800/82447
Embedded 832/82447
Embedded 864/82447
Embedded 896/82447
Embedded 928/82447
Embedded 960/82447
Embedded 992/82447
Embedded 1024/82447
Embedded 1056/82447
Embedded 1088/82447
Embedded 1120/82447
Embedded 1152/82447
Embedded 1184/82447
Embedded 1216/82447
Embedded 1248/82447
Embedded 1280/82447
Embedded 1312/82447
Embedded 1344/82447
Embedded 1376/82447
Embedded 1408/82447
Embedded 1440/82447
Embedded 1472/82447
Embedded 1504/82447
Embedded 1536/82447
Embedd

## 3. Save

In [7]:
df_save = df.copy()
df_save["embedding"] = list(embeddings)

df_save.to_parquet("jina_v3_offtheshelf_full.parquet", engine="pyarrow", index=False)